# m01_compare_models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/javanmardi298/rag_ollama/blob/main/lessons/m01_compare_models/m01_compare_models.ipynb)

**روال هر گام:** ۱) متن را بخوان ۲) کد را ببین و کامل کن (✍️) ۳) سلول را اجرا کن (Shift+Enter) ۴) «بررسی خودکار» (✅) را اجرا کن ۵) امتحان کوتاه (📝). در پایان «گزارش نهایی درس» نتیجه‌ی همه‌ی گام‌ها را نشان می‌دهد.

همه‌ی کد همین‌جا نوشته و اجرا می‌شود؛ به فایل جداگانه‌ای نیاز نیست. سلول‌ها را **به ترتیب** اجرا کن.

In [ ]:
# 1) آماده‌سازی (Colab: پروژه را clone و نصب می‌کند؛ محلی: اگر داخل پوشه‌ی پروژه‌ای چیزی تغییر نمی‌کند)
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('repo'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', 'https://github.com/javanmardi298/rag_ollama.git', 'repo'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'repo/requirements.txt', '-e', 'repo'], check=True)
if os.path.exists('repo'):
    os.chdir('repo')
while not os.path.exists('lessons') and os.getcwd() != os.path.dirname(os.getcwd()):   # باز کردن از داخل lessons/...
    os.chdir(os.path.dirname(os.getcwd()))
print('پوشه‌ی کار:', os.getcwd())

### مدل زبانی درس‌ها
پیش‌فرض `offline` است (بدون نصب؛ شکل کار را نشان می‌دهد). برای جواب واقعی یکی از خط‌های زیر را از حالت توضیح (`#`) دربیاور. کلید API را **داخل notebook ننویس** (commit می‌شود)؛ از `.env` یا `getpass` استفاده کن.

In [ ]:
import os
os.environ['LEARN_PROVIDER'] = 'offline'      # 'offline' | 'ollama' | 'anthropic' | 'opencode' | 'openai_compatible'
# os.environ['LEARN_PROVIDER'] = 'ollama';     os.environ['LEARN_MODEL'] = 'qwen2.5:7b'
# os.environ['LEARN_PROVIDER'] = 'anthropic';  os.environ['LEARN_MODEL'] = 'claude-sonnet-5-5'; os.environ['LEARN_BUDGET_USD'] = '0.5'
#   import getpass; os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')   # اشتراک Claude.ai کلید API نیست
# os.environ['LEARN_PROVIDER'] = 'opencode';   os.environ['LEARN_MODEL'] = 'anthropic/claude-sonnet-5-5'
# os.environ['LEARN_PROVIDER'] = 'openai_compatible'; os.environ['LEARN_API_BASE'] = 'http://localhost:1234/v1'; os.environ['LEARN_MODEL'] = '...'
from learn import llm as learn_llm
reply = learn_llm.get().chat([{'role': 'user', 'content': 'Reply with the single word OK.'}])
print(os.environ['LEARN_PROVIDER'], '->', reply.content[:80], f'({reply.latency_s:.1f}s)')

---

# گام ۱: تنظیم ارائه‌دهنده‌ی مدل از روی محیط (config)

## هدف این گام
تابعی می‌نویسی که متغیرهای محیطی (`LEARN_PROVIDER`، `LEARN_MODEL`، …) را می‌خواند، **اعتبارسنجی** می‌کند و یک تنظیم تمیز برمی‌گرداند. یادگیری اصلی: تنظیمات یک سیستم LLM را در **یک نقطه** جمع کن و خطای تنظیم را زود و واضح بگیر، نه وسط یک اجرای ۲۰ دقیقه‌ای.

## پیش‌نیاز
دیکشنری و استثنا در Python. [00_setup](#) را برای معنی هر ارائه‌دهنده ببین.

## مفهوم
ارائه‌دهنده‌ها فرق دارند ولی رابط ما یکی است (`chat` و `json`). تنها چیزی که عوض می‌شود **تنظیم** است:

| ارائه‌دهنده | `LEARN_MODEL` | شرط | پیش‌فرض مدل |
|---|---|---|---|
| `offline` | ندارد | هیچ | — |
| `ollama` | نام مدل | سرور Ollama | `qwen2.5:7b` |
| `anthropic` | نام مدل Claude | کلید `ANTHROPIC_API_KEY` | `claude-sonnet-5-5` |
| `opencode` | `provider/model` | OpenCode نصب باشد | ندارد (**الزامی**) |
| `openai_compatible` | نام مدل | `LEARN_API_BASE` | ندارد (**الزامی**) |

قواعد `resolve_config(env)`:
1. `provider = env.get("LEARN_PROVIDER", "offline")`؛ ناشناخته ← `ValueError("unknown provider ...")`.
2. `model = env.get("LEARN_MODEL")` یا پیش‌فرض جدول.
3. `opencode` بدون مدل، یا مدل بدون `/` ← `ValueError` (باید `provider/model` باشد).
4. `openai_compatible` بدون `LEARN_API_BASE` ← `ValueError`.
5. `anthropic` بدون کلید (`ANTHROPIC_API_KEY` در `env`) ← `ValueError` (اشتراک Claude.ai کلید نیست).
6. بودجه: `LEARN_BUDGET_USD` (پیش‌فرض ۰ = بدون سقف) به `float`؛ منفی یا غیرعددی ← `ValueError`.
7. خروجی: `{"provider", "model", "api_base", "budget_usd", "cloud": bool}` که `cloud` برای `anthropic` و `openai_compatible` درست است (داده به بیرون می‌رود).

## تمرین کد

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 1: resolve the model provider configuration from environment variables."""
from __future__ import annotations

PROVIDERS = {"offline", "ollama", "anthropic", "opencode", "openai_compatible"}
DEFAULT_MODEL = {"offline": "", "ollama": "qwen2.5:7b", "anthropic": "claude-sonnet-5-5", "opencode": "", "openai_compatible": ""}


def resolve_config(env: dict) -> dict:
    """-> {"provider", "model", "api_base", "budget_usd", "cloud"}; raises ValueError with a clear message for bad settings."""
    # TODO 1: provider from LEARN_PROVIDER (default "offline"), unknown -> ValueError
    # TODO 2: model from LEARN_MODEL or DEFAULT_MODEL
    # TODO 3: opencode needs "provider/model"; openai_compatible needs LEARN_API_BASE; anthropic needs ANTHROPIC_API_KEY in env
    # TODO 4: budget_usd = float(LEARN_BUDGET_USD or 0); not a number or negative -> ValueError
    # TODO 5: cloud = provider in {"anthropic", "openai_compatible"}
    raise NotImplementedError("TODO 1-5: resolve_config")

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('m01_compare_models', 'step01_provider_config', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
def _raises(m, env, needle):
    try:
        m.resolve_config(env)
    except ValueError as e:
        assert needle in str(e).lower(), f"message should mention {needle!r}: {e}"
        return
    raise AssertionError(f"expected ValueError for {env}")


def check_defaults(m):
    """no environment -> offline"""
    c = m.resolve_config({})
    assert c == {"provider": "offline", "model": "", "api_base": "", "budget_usd": 0.0, "cloud": False}, c


def check_ollama_default_model(m):
    """ollama gets its default model; an explicit model wins"""
    assert m.resolve_config({"LEARN_PROVIDER": "ollama"})["model"] == "qwen2.5:7b"
    assert m.resolve_config({"LEARN_PROVIDER": "ollama", "LEARN_MODEL": "llama3.2:3b"})["model"] == "llama3.2:3b"


def check_unknown_provider(m):
    """unknown provider is an error"""
    _raises(m, {"LEARN_PROVIDER": "gpt"}, "unknown provider")


def check_opencode_needs_slash(m):
    """opencode needs provider/model"""
    _raises(m, {"LEARN_PROVIDER": "opencode"}, "provider/model")
    _raises(m, {"LEARN_PROVIDER": "opencode", "LEARN_MODEL": "claude"}, "provider/model")
    assert m.resolve_config({"LEARN_PROVIDER": "opencode", "LEARN_MODEL": "anthropic/x"})["model"] == "anthropic/x"


def check_api_base_and_key(m):
    """openai_compatible needs an api base; anthropic needs a key"""
    _raises(m, {"LEARN_PROVIDER": "openai_compatible", "LEARN_MODEL": "x"}, "api_base")
    _raises(m, {"LEARN_PROVIDER": "anthropic"}, "anthropic_api_key")
    c = m.resolve_config({"LEARN_PROVIDER": "anthropic", "ANTHROPIC_API_KEY": "k"})
    assert c["cloud"] is True and c["model"] == "claude-sonnet-5-5" and "k" not in str(c["model"])


def check_budget(m):
    """budget parses to float; bad values raise"""
    assert m.resolve_config({"LEARN_BUDGET_USD": "1.5"})["budget_usd"] == 1.5
    _raises(m, {"LEARN_BUDGET_USD": "abc"}, "number")
    _raises(m, {"LEARN_BUDGET_USD": "-1"}, "negative")


def check_cloud_flag(m):
    """cloud is true only where data leaves the machine"""
    assert m.resolve_config({"LEARN_PROVIDER": "ollama"})["cloud"] is False
    assert m.resolve_config({"LEARN_PROVIDER": "openai_compatible", "LEARN_API_BASE": "http://x/v1"})["cloud"] is True
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 1: resolve the model provider configuration from environment variables."""
from __future__ import annotations

PROVIDERS = {"offline", "ollama", "anthropic", "opencode", "openai_compatible"}
DEFAULT_MODEL = {"offline": "", "ollama": "qwen2.5:7b", "anthropic": "claude-sonnet-5-5", "opencode": "", "openai_compatible": ""}


def resolve_config(env: dict) -> dict:
    """-> {"provider", "model", "api_base", "budget_usd", "cloud"}; raises ValueError with a clear message for bad settings."""
    provider = env.get("LEARN_PROVIDER", "offline")
    if provider not in PROVIDERS:
        raise ValueError(f"unknown provider {provider!r}; choose one of {sorted(PROVIDERS)}")
    model = env.get("LEARN_MODEL") or DEFAULT_MODEL[provider]
    api_base = env.get("LEARN_API_BASE", "")
    if provider == "opencode" and "/" not in model:
        raise ValueError("opencode needs LEARN_MODEL=provider/model (see `opencode models`)")
    if provider == "openai_compatible" and not api_base:
        raise ValueError("openai_compatible needs LEARN_API_BASE, e.g. http://localhost:1234/v1")
    if provider == "anthropic" and not env.get("ANTHROPIC_API_KEY"):
        raise ValueError("anthropic needs ANTHROPIC_API_KEY (an API key from console.anthropic.com; a Claude.ai subscription is not one)")
    try:
        budget = float(env.get("LEARN_BUDGET_USD") or 0)
    except ValueError:
        raise ValueError("LEARN_BUDGET_USD must be a number") from None
    if budget < 0:
        raise ValueError("LEARN_BUDGET_USD must not be negative")
    return {"provider": provider, "model": model, "api_base": api_base, "budget_usd": budget,
            "cloud": provider in {"anthropic", "openai_compatible"}}
```

</details>

## خروجی مورد انتظار
هفت ✓.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| `float("abc")` استثنای دیگری داد | `ValueError` خودت را با پیام واضح بالا بیاور |
| مدل پیش‌فرض ollama خالی است | از جدول پیش‌فرض‌ها استفاده نکرده‌ای |

## دام‌ها
- هرگز **مقدار** کلید را چاپ یا لاگ نکن؛ فقط وجود آن را بررسی کن.
- پیش‌فرض‌های مدل با زمان کهنه می‌شوند؛ نام مدل‌های فعلی را از مستندات هر ارائه‌دهنده بگیر.

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('m01_compare_models', 'step01_provider_config')

---

# گام ۲: آزمون قابل‌تکرار — دقت، تأخیر، خطا

## هدف این گام
تابع `run_benchmark` را می‌نویسی که چند مدل (هر کدام با رابط `chat`) را روی **همان سؤال‌ها** اجرا می‌کند و برای هر مدل دقت، میانگین تأخیر و تعداد خطاها را برمی‌گرداند.

## پیش‌نیاز
گام ۱ مفهوماً. این تابع به هیچ ارائه‌دهنده‌ی مشخصی وابسته نیست؛ هر شیئی با `.chat(messages).content` کافی است (در تست‌ها مدل ساختگی).

## مفهوم
یک آزمون خوب سه ویژگی دارد:
1. **یکسان:** همه‌ی مدل‌ها همان prompt ها را می‌بینند.
2. **خودکار:** درست/غلط را **کد** تعیین می‌کند، نه چشم تو: هر مورد یک تابع `check(answer) -> bool` دارد. (برای پرسش‌های باز، LLM-judge لازم است و آن را باید اعتبارسنجی کنی؛ اینجا پاسخ‌های کوتاه و قطعی می‌پرسیم.)
3. **مقاوم به شکست:** یک خطای شبکه یا زمان‌گذشت یک مدل نباید کل آزمون را بکشد؛ آن را **به‌عنوان خطا بشمار** و ادامه بده.

هر ردیف خروجی: `{"model", "accuracy", "mean_latency_s", "errors", "n"}`.
- `accuracy` = درست‌ها / `n` (خطا غلط حساب می‌شود).
- `mean_latency_s` = میانگین زمان فراخوانی‌های **موفق** (اگر هیچ موفقیتی نبود `None`).
- زمان را با `time.perf_counter()` بگیر.

## تمرین کد

`run_benchmark(models, cases, repeats=1)`:
- `models`: دیکشنری `نام ← شیء llm`.
- `cases`: لیست `{"prompt": str, "check": callable}`.
- هر مدل × هر مورد × `repeats` بار. پیام: `[{"role": "user", "content": prompt}]`.
- خروجی: لیست ردیف‌ها به ترتیب `models`.

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 2: the same cases on every model: accuracy, latency, errors."""
from __future__ import annotations

import os
import time

CASES = [
    {"prompt": "What is 17 + 25? Answer with the number only.", "check": lambda a: "42" in a},
    {"prompt": "What is the capital of France? One word.", "check": lambda a: "paris" in a.lower()},
    {"prompt": "Translate 'good morning' to German. Two words.", "check": lambda a: "guten morgen" in a.lower()},
    {"prompt": "Return only the word YES if 9 is greater than 3, otherwise NO.", "check": lambda a: a.strip().upper().startswith("YES")},
]


def run_benchmark(models: dict, cases: list[dict], repeats: int = 1) -> list[dict]:
    """models: name -> object with .chat(messages).content. Returns one row per model:
    {"model", "accuracy", "mean_latency_s", "errors", "n"}"""
    # TODO 1: for every model, case and repeat: time models[name].chat([{"role": "user", "content": prompt}]).content
    #         with time.perf_counter(); an exception counts as an error AND as a wrong answer (never stop the run);
    #         accuracy = correct / n; mean_latency_s = mean over the successful calls only (None if there were none)
    raise NotImplementedError("TODO 1: run_benchmark")


def build_models(spec: str) -> dict:
    """'offline,ollama:qwen2.5:7b,anthropic:claude-sonnet-5-5' -> {name: llm}; unavailable ones are skipped with a reason."""
    from learn import llm as learn_llm
    out = {}
    for item in filter(None, (x.strip() for x in spec.split(","))):
        provider, _, model = item.partition(":")
        try:
            out[item] = learn_llm.get(provider, model)
        except Exception as e:
            print(f"skipped {item}: {e}")
    return out

### ▶ اجرای نمایشی
اول کد را بخوان، بعد اجرا کن. این سلول از توابعی که بالا نوشتی استفاده می‌کند.

In [ ]:
models = build_models(os.environ.get("COMPARE", "offline"))
for r in run_benchmark(models, CASES):
    print(r)

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('m01_compare_models', 'step02_benchmark', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
import time


class Reply:
    def __init__(self, content):
        self.content = content


class Fixed:
    def __init__(self, answers, delay=0.0, fail_on=()):
        self.answers, self.delay, self.fail_on, self.i = answers, delay, set(fail_on), 0

    def chat(self, messages, tools=None, json_schema=None):
        i, self.i = self.i, self.i + 1
        time.sleep(self.delay)
        if i in self.fail_on:
            raise RuntimeError("boom")
        return Reply(self.answers[i % len(self.answers)])


CASES = [{"prompt": "a", "check": lambda x: x == "A"}, {"prompt": "b", "check": lambda x: x == "B"}]


def check_accuracy(m):
    """accuracy = correct / n"""
    rows = m.run_benchmark({"good": Fixed(["A", "B"]), "half": Fixed(["A", "x"])}, CASES)
    assert [r["model"] for r in rows] == ["good", "half"]
    assert rows[0]["accuracy"] == 1.0 and rows[1]["accuracy"] == 0.5 and rows[0]["n"] == 2, rows


def check_errors_do_not_stop(m):
    """an exception is counted, the run continues"""
    rows = m.run_benchmark({"flaky": Fixed(["A", "B"], fail_on={0})}, CASES)
    assert rows[0]["errors"] == 1 and rows[0]["accuracy"] == 0.5 and rows[0]["n"] == 2, rows


def check_latency_only_successes(m):
    """mean latency counts successful calls only"""
    rows = m.run_benchmark({"slow": Fixed(["A", "B"], delay=0.02)}, CASES)
    assert 0.015 < rows[0]["mean_latency_s"] < 0.2, rows
    allfail = m.run_benchmark({"dead": Fixed(["A"], fail_on={0, 1})}, CASES)
    assert allfail[0]["mean_latency_s"] is None and allfail[0]["errors"] == 2 and allfail[0]["accuracy"] == 0.0, allfail


def check_repeats(m):
    """repeats multiplies the number of calls"""
    llm = Fixed(["A", "B"])
    rows = m.run_benchmark({"x": llm}, CASES, repeats=3)
    assert rows[0]["n"] == 6 and llm.i == 6, (rows, llm.i)


def check_same_prompts_for_all(m):
    """every model gets the same user message format"""
    seen = []

    class Spy(Fixed):
        def chat(self, messages, tools=None, json_schema=None):
            seen.append(messages)
            return super().chat(messages)
    m.run_benchmark({"s": Spy(["A", "B"])}, CASES)
    assert seen == [[{"role": "user", "content": "a"}], [{"role": "user", "content": "b"}]], seen


def check_empty_cases(m):
    """no cases -> accuracy 0.0 and no crash"""
    rows = m.run_benchmark({"x": Fixed(["A"])}, [])
    assert rows[0]["n"] == 0 and rows[0]["accuracy"] == 0.0 and rows[0]["mean_latency_s"] is None, rows
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 2: the same cases on every model: accuracy, latency, errors."""
from __future__ import annotations

import os
import time

CASES = [
    {"prompt": "What is 17 + 25? Answer with the number only.", "check": lambda a: "42" in a},
    {"prompt": "What is the capital of France? One word.", "check": lambda a: "paris" in a.lower()},
    {"prompt": "Translate 'good morning' to German. Two words.", "check": lambda a: "guten morgen" in a.lower()},
    {"prompt": "Return only the word YES if 9 is greater than 3, otherwise NO.", "check": lambda a: a.strip().upper().startswith("YES")},
]


def run_benchmark(models: dict, cases: list[dict], repeats: int = 1) -> list[dict]:
    """models: name -> object with .chat(messages).content. Returns one row per model:
    {"model", "accuracy", "mean_latency_s", "errors", "n"}"""
    rows = []
    for name, llm in models.items():
        correct = errors = 0
        times: list[float] = []
        n = 0
        for case in cases:
            for _ in range(repeats):
                n += 1
                start = time.perf_counter()
                try:
                    answer = llm.chat([{"role": "user", "content": case["prompt"]}]).content
                except Exception:
                    errors += 1
                    continue
                times.append(time.perf_counter() - start)
                correct += bool(case["check"](answer))
        rows.append({"model": name, "accuracy": correct / n if n else 0.0, "mean_latency_s": sum(times) / len(times) if times else None,
                     "errors": errors, "n": n})
    return rows


def build_models(spec: str) -> dict:
    """'offline,ollama:qwen2.5:7b,anthropic:claude-sonnet-5-5' -> {name: llm}; unavailable ones are skipped with a reason."""
    from learn import llm as learn_llm
    out = {}
    for item in filter(None, (x.strip() for x in spec.split(","))):
        provider, _, model = item.partition(":")
        try:
            out[item] = learn_llm.get(provider, model)
        except Exception as e:
            print(f"skipped {item}: {e}")
    return out
```

</details>

## خروجی مورد انتظار
شش ✓. اجرای دستی با `offline` دقت پایین نشان می‌دهد (مدل ساختگی پاسخ‌ها را نمی‌داند): این درست است. با دو مدل واقعی، جدولی از دقت و تأخیر می‌بینی.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| یک خطا کل آزمون را می‌کشد | `try/except` دور هر فراخوانی نگذاشته‌ای |
| خطا به‌عنوان درست شمرده شد | در حالت خطا `ok=False` باید باشد |
| تأخیر خطاها هم در میانگین است | فقط موفق‌ها را بشمار |

## دام‌ها
- **ده سؤال یعنی اختلاف ۱۰٪ فقط یک سؤال است.** با آزمون کوچک نتیجه‌ی قطعی نگیر؛ مدل‌ها را روی ۳۰ تا ۱۰۰ مورد **از داده‌ی خودت** بسنج.
- temperature را صفر نگه دار و `repeats` را برای مدل‌های تصادفی بیشتر کن.
- اولین فراخوانی Ollama مدل را در حافظه بار می‌کند و کند است؛ یک فراخوانی گرم‌کردن (warm-up) را در تأخیر نشمار.
- تأخیر ابری شامل شبکه است؛ تأخیر محلی به سخت‌افزار تو بستگی دارد. مقایسه‌ی «منصفانه» نیست، فقط مقایسه‌ی «عملی» برای مورد استفاده‌ی توست.

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('m01_compare_models', 'step02_benchmark')

---

# گام ۳: انتخاب مدل با قاعده و گزارش قابل‌انتشار

## هدف این گام
از ردیف‌های آزمون، با یک **قاعده‌ی از پیش تعیین‌شده** مدل را انتخاب می‌کنی (`choose`) و جدول markdown می‌سازی (`markdown_table`) که مستقیم در README پروژه یا رزومه‌ات می‌گذاری.

## پیش‌نیاز
خروجی گام ۲ (لیست ردیف‌ها). تست‌ها ردیف‌های آماده می‌دهند.

## مفهوم
«بهترین مدل» بدون **معیار** معنا ندارد. معیار را **قبل از دیدن نتیجه** بنویس:

> «از بین مدل‌هایی که دقتشان ≥ ۰٫۸ است و خطایی ندارند، کم‌تأخیرترین را بردار. اگر هیچ‌کدام نبود، هیچ.»

این قاعده سه ویژگی خوب دارد: ساده است، قابل‌تکرار است، و از «تنظیم معیار بعد از دیدن نتیجه» (که تقلب خودخواسته است) جلوگیری می‌کند. اگر هیچ مدلی نرسید، **جواب `None` است** و باید مدل بهتر بگیری یا کار را آسان‌تر کنی؛ نباید آستانه را پایین بیاوری تا یکی «برنده» شود.

`choose(rows, min_accuracy=0.8, max_errors=0)`:
- فقط ردیف‌هایی که `accuracy >= min_accuracy` و `errors <= max_errors` و `mean_latency_s` غیر `None` دارند.
- از بین آن‌ها کمترین `mean_latency_s`؛ اگر مساوی بود، بیشترین دقت؛ اگر باز مساوی بود، نام به ترتیب الفبا.
- خروجی: **نام** مدل یا `None`.

`markdown_table(rows)`:
```
| model | accuracy | latency (s) | errors |
|---|---|---|---|
| ollama:qwen2.5:7b | 0.75 | 1.20 | 0 |
```
- ردیف‌ها به ترتیب دقت نزولی (مساوی ← نام).
- دقت با ۲ رقم اعشار؛ تأخیر با ۲ رقم یا `-` اگر `None`.

## تمرین کد

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 3: choose a model with a rule fixed in advance, and print a report table."""
from __future__ import annotations


def choose(rows: list[dict], min_accuracy: float = 0.8, max_errors: int = 0):
    """Among rows with accuracy >= min_accuracy, errors <= max_errors and a measured latency, return the NAME of the fastest
    (ties: higher accuracy, then name). None if no row qualifies."""
    # TODO 1: filter, then min(..., key=(latency, -accuracy, name))
    raise NotImplementedError("TODO 1: choose")


def markdown_table(rows: list[dict]) -> str:
    """'| model | accuracy | latency (s) | errors |' header, separator, then rows sorted by accuracy descending (ties: name).
    accuracy with 2 decimals; latency with 2 decimals or '-' when None."""
    # TODO 2: build the lines and join them with "\n"
    raise NotImplementedError("TODO 2: markdown_table")

### ▶ اجرای نمایشی
اول کد را بخوان، بعد اجرا کن. این سلول از توابعی که بالا نوشتی استفاده می‌کند.

In [ ]:
rows = [{"model": "ollama:qwen2.5:7b", "accuracy": 0.75, "mean_latency_s": 1.2, "errors": 0, "n": 4},
        {"model": "anthropic:claude", "accuracy": 1.0, "mean_latency_s": 0.9, "errors": 0, "n": 4},
        {"model": "opencode:x/y", "accuracy": 1.0, "mean_latency_s": 3.0, "errors": 0, "n": 4}]
print(markdown_table(rows))
print("chosen:", choose(rows))

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('m01_compare_models', 'step03_choose_and_report', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
ROWS = [{"model": "local", "accuracy": 0.75, "mean_latency_s": 1.2, "errors": 0, "n": 4},
        {"model": "cloud_fast", "accuracy": 1.0, "mean_latency_s": 0.9, "errors": 0, "n": 4},
        {"model": "cloud_slow", "accuracy": 1.0, "mean_latency_s": 3.0, "errors": 0, "n": 4},
        {"model": "flaky", "accuracy": 1.0, "mean_latency_s": 0.1, "errors": 1, "n": 4},
        {"model": "dead", "accuracy": 0.0, "mean_latency_s": None, "errors": 4, "n": 4}]


def check_choose_fastest_qualified(m):
    """fastest among models that meet the rule (flaky and local are out)"""
    assert m.choose(ROWS) == "cloud_fast", m.choose(ROWS)


def check_choose_none(m):
    """nobody qualifies -> None"""
    assert m.choose(ROWS, min_accuracy=1.1) is None
    assert m.choose([]) is None


def check_choose_threshold_parameters(m):
    """thresholds are parameters"""
    assert m.choose(ROWS, min_accuracy=0.7) == "cloud_fast"
    assert m.choose(ROWS, min_accuracy=1.0, max_errors=1) == "flaky"


def check_tie_breaks(m):
    """tie on latency: higher accuracy, then name"""
    rows = [{"model": "b", "accuracy": 0.9, "mean_latency_s": 1.0, "errors": 0, "n": 1},
            {"model": "a", "accuracy": 0.9, "mean_latency_s": 1.0, "errors": 0, "n": 1},
            {"model": "c", "accuracy": 1.0, "mean_latency_s": 1.0, "errors": 0, "n": 1}]
    assert m.choose(rows) == "c"
    assert m.choose(rows[:2]) == "a"


def check_table_format(m):
    """header, separator, sorted rows, 2 decimals, '-' for missing latency"""
    t = m.markdown_table(ROWS).splitlines()
    assert t[0] == "| model | accuracy | latency (s) | errors |" and t[1] == "|---|---|---|---|", t[:2]
    assert t[2] == "| cloud_fast | 1.00 | 0.90 | 0 |", t[2]
    assert t[-1] == "| dead | 0.00 | - | 4 |", t[-1]


def check_table_sorting(m):
    """sorted by accuracy descending, ties by name"""
    names = [line.split("|")[1].strip() for line in m.markdown_table(ROWS).splitlines()[2:]]
    assert names == ["cloud_fast", "cloud_slow", "flaky", "local", "dead"], names
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 3: choose a model with a rule fixed in advance, and print a report table."""
from __future__ import annotations


def choose(rows: list[dict], min_accuracy: float = 0.8, max_errors: int = 0):
    """Among rows with accuracy >= min_accuracy, errors <= max_errors and a measured latency, return the NAME of the fastest
    (ties: higher accuracy, then name). None if no row qualifies."""
    ok = [r for r in rows if r["accuracy"] >= min_accuracy and r["errors"] <= max_errors and r["mean_latency_s"] is not None]
    if not ok:
        return None
    return min(ok, key=lambda r: (r["mean_latency_s"], -r["accuracy"], r["model"]))["model"]


def markdown_table(rows: list[dict]) -> str:
    """'| model | accuracy | latency (s) | errors |' header, separator, then rows sorted by accuracy descending (ties: name).
    accuracy with 2 decimals; latency with 2 decimals or '-' when None."""
    lines = ["| model | accuracy | latency (s) | errors |", "|---|---|---|---|"]
    for r in sorted(rows, key=lambda r: (-r["accuracy"], r["model"])):
        lat = "-" if r["mean_latency_s"] is None else f"{r['mean_latency_s']:.2f}"
        lines.append(f"| {r['model']} | {r['accuracy']:.2f} | {lat} | {r['errors']} |")
    return "\n".join(lines)
```

</details>

## خروجی مورد انتظار
شش ✓ و چاپ یک جدول نمونه.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| `TypeError` با `None` | مدلی بدون تأخیر (`None`) را قبل از مقایسه حذف نکرده‌ای |
| ترتیب جدول عوض می‌شود | مرتب‌سازی با کلید ترکیبی `(-accuracy, model)` |
| `-` چاپ نمی‌شود | حالت `None` را فرمت نکرده‌ای |

## دام‌ها
- هزینه‌ی ابری هم بخشی از انتخاب است: ستون «هزینه به‌ازای ۱۰۰۰ درخواست» را از قیمت واقعی خودت اضافه کن (تمرین).
- مدل برنده را در `config.yaml` پروژه ثبت کن و **تاریخ و نسخه‌ی** آزمون را کنارش بنویس؛ مدل‌ها و قیمت‌ها عوض می‌شوند.
- به نتیجه‌ی ۱۰ مورد تصمیم تجاری نگیر (گام ۲).

## جمع‌بندی m01 و قدم بعد
حالا می‌توانی بگویی «برای کار من مدل X را انتخاب کردم، چون روی ۵۰ نمونه‌ی خودم دقت … و تأخیر … داشت، با هزینه‌ی …». قدم بعد: همین آزمون را با داده‌ی خودت و `llm_judge` اعتبارسنجی‌شده برای خروجی‌های باز، و ثبت نتیجه در پنل (`results/`).

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('m01_compare_models', 'step03_choose_and_report')

---
## 🏁 گزارش نهایی درس
نتیجه‌ی تست و امتحان هر گام، و آنچه هنوز مانده.

In [ ]:
from learn.notebook import lesson_report
lesson_report('m01_compare_models')